**<h1>1. Tokenizer pré-entraîné sur HuggingFace</h1>**

In [ ]:
# =========================================================
# 1. INSTALLATION (si nécessaire)
# =========================================================
# pip install tensorflow transformers datasets scikit-learn

import tensorflow as tf
import numpy as np
from datasets import load_dataset
from transformers import BertTokenizer
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [ ]:
# =========================================================
# 2. CHOIX DATASET (PETIT OU COMPLET)
# =========================================================

USE_SMALL_DATASET = True   # changer à False pour tout le dataset

# IMDB dataset (sentiment analysis)
dataset = load_dataset("tblard/allocine")

if USE_SMALL_DATASET:
    train_data = dataset["train"].select(range(2000))
    test_data = dataset["test"].select(range(1000))
else:
    train_data = dataset["train"]
    test_data = dataset["test"]

print("Train size:", len(train_data))
print("Test size:", len(test_data))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

allocine/train-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

allocine/validation-00000-of-00001.parqu(…):   0%|          | 0.00/7.58M [00:00<?, ?B/s]

allocine/test-00000-of-00001.parquet:   0%|          | 0.00/7.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Train size: 2000
Test size: 1000


In [ ]:
# =========================================================
# 3. TOKENIZER BERT (préentraîné)
# =========================================================

from transformers import AutoModelForMaskedLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained("almanach/camembert-large")

MAX_LEN = 128

def tokenize_function(example):
    return tokenizer(
        example["review"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

# Tokenisation dataset
train_encodings = train_data.map(tokenize_function)
test_encodings = test_data.map(tokenize_function)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# 4. PREPARATION X / Y
# =========================================================

def extract_features(data):
    X = np.array(data["input_ids"])
    y = np.array(data["label"])
    return X, y

X_train, y_train = extract_features(train_encodings)
X_test, y_test = extract_features(test_encodings)

print("X_train shape:", X_train.shape)


X_train shape: (2000, 128)


In [ ]:
# =========================================================
# 5. MODELE RNN (LSTM) AVEC KERAS FUNCTIONAL API
# =========================================================

VOCAB_SIZE = tokenizer.vocab_size
MAX_LEN = 128

# -------------------------
# INPUT
# -------------------------
inputs = Input(shape=(MAX_LEN,), name="input_ids")

# -------------------------
# EMBEDDING
# -------------------------
x = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=128,
    name="embedding"
)(inputs)

# -------------------------
# LSTM STACK
# -------------------------
x = SimpleRNN(64, return_sequences=True, name="rnn_1")(x)
x = SimpleRNN(32, name="rnn_2")(x)

# -------------------------
# OUTPUT
# -------------------------
outputs = Dense(1, activation="sigmoid", name="output")(x)

# -------------------------
# MODEL
# -------------------------
model = Model(inputs=inputs, outputs=outputs, name="RNN_Model")

# -------------------------
# COMPILE
# -------------------------
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "RNN_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_ids (InputLayer)          │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 128, 128)       │     4,096,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_1 (SimpleRNN)               │ (None, 128, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_2 (SimpleRNN)               │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,112,129 (15.69 MB)

 Trainable params: 4,112,129 (15.69 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# =========================================================
# 6. TRAINING
# =========================================================

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=32
)

Epoch 1/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 135ms/step - accuracy: 0.4931 - loss: 0.7056 - val_accuracy: 0.5025 - val_loss: 0.6949
Epoch 2/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 216ms/step - accuracy: 0.7169 - loss: 0.5258 - val_accuracy: 0.4925 - val_loss: 0.7527
Epoch 3/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 90ms/step - accuracy: 0.7631 - loss: 0.4038 - val_accuracy: 0.5000 - val_loss: 0.8491


In [ ]:
# =========================================================
# 7. EVALUATION
# =========================================================

loss, acc = model.evaluate(X_test, y_test)
print("Test accuracy:", acc)


32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5210 - loss: 0.8468
Test accuracy: 0.5210000276565552


In [ ]:
# =========================================================
# 8. TEST PRÉDICTION
# =========================================================

def predict_text(text):
    tokens = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="np"
    )

    pred = model.predict(tokens["input_ids"])[0][0]

    label = "Positif" if pred > 0.5 else "Négatif"

    return label, float(pred)

# Exemple
print(predict_text("This movie was really amazing and exciting"))
print(predict_text("I did not like this movie at all"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step
('Négatif', 0.44655898213386536)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
('Négatif', 0.4151187837123871)


In [ ]:
# =========================================
# Sauvegarder le modèle
# ========================================

model.save_weights("model_allocine.weights.h5")

In [ ]:
tokenizer.vocab

{'univers': 2302,
 'icône': 12184,
 '▁propre': 731,
 '▁inapproprié': 26676,
 '▁tempête': 8686,
 '▁existait': 16403,
 '▁imaginez': 25573,
 '▁Seigneur': 2745,
 'brac': 21152,
 '▁résilience': 21112,
 '▁pouvez': 425,
 'rien': 8246,
 '▁Messieurs': 19380,
 '▁Beth': 25230,
 '▁Bourges': 20828,
 '▁2014)': 12907,
 'Sol': 17992,
 '6-4': 30230,
 '▁BCE': 18331,
 '▁PA': 8185,
 'Equipe': 20734,
 '▁passionnés': 7675,
 '▁formule': 3439,
 '▁spécialement': 5354,
 'ds': 11280,
 '▁Simone': 13700,
 'Ou': 9436,
 '▁bande': 1965,
 '▁récupérer': 4595,
 '▁installer': 5554,
 'Sport': 19111,
 '▁voleur': 16164,
 '▁Rio': 7157,
 '▁Parmi': 2462,
 '▁Allemagne': 3021,
 '▁imposable': 19118,
 '▁bienveillante': 24862,
 '▁apparaît': 3433,
 '440': 24900,
 '▁opaque': 25791,
 'GIE': 18094,
 '▁occasions': 11611,
 'ette': 1451,
 '▁Fontaine': 9938,
 '▁combiné': 13915,
 '▁Golf': 13662,
 'px': 16160,
 '▁douleur': 3420,
 '▁patrimoine': 1369,
 '▁saveurs': 10111,
 'ku': 8235,
 '▁élémentaires': 19630,
 '▁insectes': 11002,
 '▁politiquem

In [ ]:
len(tokenizer.vocab)

32005

**<h1>2. TP : Analyse de texte avec RNN et déploiement fullstack</h1>**


**<h3>Objectif du TP</h3>**

À partir de l'exemple basé sur le dataset IMDB, vous devez :

1. choisir un dataset de classification de texte disponible sur Hugging Face
2. entraîner un modèle de réseau de neurones récurrent (RNN)
3. utiliser un tokenizer préentraîné (BERT) pour transformer le texte en séquences numériques
4. créer une application fullstack permettant d’interroger le modèle :

   * backend : Flask (API de prédiction)
   * frontend : Streamlit (interface chatbot)

L’objectif est de comprendre le pipeline complet d’un système d’analyse de texte basé sur le deep learning.


**<h2> 1. Dataset IMDB (référence de départ)</h2>**

[https://huggingface.co/datasets/imdb](https://huggingface.co/datasets/stanfordnlp/imdb)

**<h3>Description</h3>**

Le dataset IMDB contient des critiques de films labellisées :

* 0 : négatif
* 1 : positif

**<h3>Intérêt pédagogique</h3>**

* problème simple de classification binaire
* idéal pour comprendre le RNN
* dataset de référence en NLP


**<h2> 2. Datasets proposés (choisir un seul pour le TP)</h2>**


**<h3>2.1 AG News</h3>**

[https://huggingface.co/datasets/ag_news](https://huggingface.co/datasets/sh0416/ag_news)

**<h4>Description</h4>**

Dataset de classification de textes journalistiques en 4 catégories :

* World
* Sports
* Business
* Sci/Tech

**<h4>Intérêt pédagogique</h4>**

* classification multi-classe
* textes courts et propres
* très adapté aux RNN


**<h3>2.2 Yelp Reviews</h3>**

[https://huggingface.co/datasets/yelp_review_full](https://huggingface.co/datasets/Yelp/yelp_review_full)

**<h4>Description</h4>**

Dataset d’avis clients notés de 1 à 5 étoiles.

**<h4>Intérêt pédagogique</h4>**

* classification multi-classe ordinale
* données réalistes issues du monde réel
* permet d’étudier des sentiments nuancés


**<h3>2.3 BBC News</h3>**

[https://huggingface.co/datasets/bbc-news](https://huggingface.co/datasets/SetFit/bbc-news)

**<h4>Description</h4>**

Dataset de classification de documents en 5 catégories :

* business
* entertainment
* politics
* sport
* tech

**<h4>Intérêt pédagogique</h4>**

* dataset équilibré
* bon compromis difficulté / performance
* adapté aux modèles RNN


**<h3>2.4 SST-2 (GLUE)</h3>**

[https://huggingface.co/datasets/glue](https://huggingface.co/datasets/nyu-mll/glue)

**<h4>Description</h4>**

Sous-dataset SST-2 :

* phrases courtes
* sentiment positif / négatif

**<h4>Intérêt pédagogique</h4>**

* données plus naturelles que IMDB
* plus difficile car phrases courtes
* bon test de généralisation


**<h2>3. Travail demandé</h2>**

**<h3>Étape 1 : Prétraitement</h3>**

* charger le dataset Hugging Face choisi
* appliquer un tokenizer BERT (`bert-base-uncased`)
* convertir les textes en séquences numériques


**<h3>Étape 2 : Modèle RNN</h3>**

* construire un modèle Keras avec API `Model`
* architecture recommandée :

  * Embedding
  * RNN
  * Dense (classification)


**<h3>Étape 3 : Entraînement</h3>**

* entraîner le modèle sur un sous-ensemble si nécessaire
* évaluer les performances


**<h3>Étape 4 : Backend Flask</h3>**

Créer une API REST qui :

* reçoit un texte
* applique le tokenizer
* prédit le sentiment ou la classe
* retourne le résultat


**<h3>Étape 5 : Frontend Streamlit</h3>**

Créer une interface chatbot :

* saisie de texte
* affichage du résultat
* historique des échanges
* affichage dynamique du label prédit


In [ ]:
# Créer un env virtuel
! python -m venv venv


# Installer les packages
! pip install tensorflow flask streamlit numpy pandas transformers datasets

# créer le requirements
! pip freeze > requirements.txt

```
sentiment-chatbot/
│
├── backend/
│   ├── app.py
│   ├── model.weights.h5
│   ├── word_index.pkl
│
├── frontend/
│   ├── streamlit_app.py
│
├── requirements.txt
└── venv/
```
